# Chapter 11: AI Agent with Tools

**Build an AI agent with three tools — web search, calculator, database query — and watch it reason, plan, and execute multi-step tasks.**

An "agent" is just a loop: the LLM thinks about what to do, picks a tool, observes the
result, and repeats until the goal is achieved. In this notebook we build a `SimpleAgent`
from scratch using OpenAI function calling, add guardrails, and trace every step.

**Key insight**: Agents are workflow engines with LLM-driven routing.

---

## Setup

Install dependencies and configure your OpenAI API key.

In [ ]:
# Install dependencies (run once)
!pip install -q openai tabulate

In [ ]:
import os
import json
import math
from typing import Any
from tabulate import tabulate

# Set your API key as an environment variable before running.
# In Colab: use the Secrets sidebar (key icon) or:
#   import os; os.environ["OPENAI_API_KEY"] = "sk-..."
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY not set. The agent will not be able to run.")
    print("Set it via: os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print("API key configured.")

## 1. Define Mock Tools

We create three tools the agent can call. In production these would hit real APIs;
here we use deterministic mock data so the notebook is self-contained and free to run.

In [ ]:
# ── Tool 1: Product Search ──────────────────────────────────────────
PRODUCT_CATALOG = [
    {"id": "P001", "name": "Basic Widget", "price": 12.99, "category": "widgets"},
    {"id": "P002", "name": "Pro Widget", "price": 29.99, "category": "widgets"},
    {"id": "P003", "name": "Widget Deluxe", "price": 59.99, "category": "widgets"},
    {"id": "P004", "name": "Gadget Mini", "price": 9.99, "category": "gadgets"},
    {"id": "P005", "name": "Gadget Standard", "price": 24.99, "category": "gadgets"},
    {"id": "P006", "name": "Gadget Pro", "price": 74.99, "category": "gadgets"},
    {"id": "P007", "name": "Connector Cable", "price": 4.99, "category": "accessories"},
    {"id": "P008", "name": "Carrying Case", "price": 19.99, "category": "accessories"},
]

def search_products(query: str) -> str:
    """Search the product catalog. Returns matching products as JSON."""
    q = query.lower()
    matches = [
        p for p in PRODUCT_CATALOG
        if q in p["name"].lower() or q in p["category"].lower()
    ]
    if not matches:
        # Fallback: return all products if no keyword match
        matches = PRODUCT_CATALOG
    return json.dumps(matches, indent=2)


# ── Tool 2: Calculator ──────────────────────────────────────────────
SAFE_MATH_NAMES = {
    "abs": abs, "round": round, "min": min, "max": max,
    "sum": sum, "pow": pow, "sqrt": math.sqrt, "pi": math.pi,
}

def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression. Returns the result as a string."""
    try:
        # Only allow safe math builtins — no file/OS access
        result = eval(expression, {"__builtins__": {}}, SAFE_MATH_NAMES)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# ── Tool 3: Customer Lookup ─────────────────────────────────────────
CUSTOMER_DB = {
    "C001": {"name": "Alice Johnson", "tier": "gold", "interests": ["widgets", "accessories"],
             "total_spent": 349.50, "last_purchase": "Pro Widget"},
    "C002": {"name": "Bob Smith", "tier": "silver", "interests": ["gadgets"],
             "total_spent": 124.98, "last_purchase": "Gadget Standard"},
    "C003": {"name": "Carol Davis", "tier": "bronze", "interests": ["accessories"],
             "total_spent": 24.98, "last_purchase": "Connector Cable"},
}

def lookup_customer(customer_id: str) -> str:
    """Look up a customer by ID. Returns customer info as JSON."""
    customer = CUSTOMER_DB.get(customer_id.upper())
    if customer:
        return json.dumps({"id": customer_id.upper(), **customer}, indent=2)
    return json.dumps({"error": f"Customer {customer_id} not found."})


# Registry: maps function names to callables and their OpenAI tool schemas
TOOL_REGISTRY = {
    "search_products": search_products,
    "calculate": calculate,
    "lookup_customer": lookup_customer,
}

print("Tools defined: search_products, calculate, lookup_customer")
print(f"Product catalog: {len(PRODUCT_CATALOG)} items")
print(f"Customer DB: {len(CUSTOMER_DB)} customers")

## 2. OpenAI Tool Schemas

We define the function-calling schemas that tell the LLM what tools are available
and what arguments they accept.

In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search the product catalog by keyword. Returns matching products with id, name, price, and category.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search keyword (e.g., 'widget', 'gadget', 'accessories')"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Supports basic arithmetic, sqrt, pow, abs, round, min, max.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '3 * 12.99 * 1.085'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_customer",
            "description": "Look up a customer by their ID (e.g., 'C001'). Returns name, tier, interests, spending history.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "The customer ID, e.g. 'C001'"
                    }
                },
                "required": ["customer_id"]
            }
        }
    }
]

print(f"Registered {len(TOOL_SCHEMAS)} tool schemas.")

## 3. Build the SimpleAgent

The agent implements the classic **think → act → observe** loop:

1. **Think**: Send the conversation to the LLM. It either generates a final answer or
   requests a tool call.
2. **Act**: Execute the requested tool with the provided arguments.
3. **Observe**: Feed the tool's output back into the conversation as a `tool` message.

Repeat until the LLM produces a final answer (no tool call) or we hit the step budget.

In [ ]:
from openai import OpenAI

class SimpleAgent:
    """A minimal agent that uses OpenAI function calling to reason and act."""

    def __init__(self, model: str = "gpt-4o-mini", max_steps: int = 5):
        self.client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
        self.model = model
        self.max_steps = max_steps
        self.tool_allowlist = set(TOOL_REGISTRY.keys())  # guardrail
        self.execution_trace = []  # log of every step
        self.total_tokens_used = 0

    def _call_tool(self, name: str, arguments: dict) -> str:
        """Execute a tool by name. Enforces the allowlist guardrail."""
        if name not in self.tool_allowlist:
            return f"Error: tool '{name}' is not in the allowlist."
        func = TOOL_REGISTRY[name]
        # Pass arguments as keyword args
        return func(**arguments)

    def run(self, goal: str) -> dict:
        """Execute the agent loop for the given goal."""
        self.execution_trace = []
        self.total_tokens_used = 0

        if not self.client:
            return {"answer": "ERROR: No API key set.", "steps": 0, "trace": []}

        messages = [
            {"role": "system", "content": (
                "You are a helpful assistant with access to tools. "
                "Use the tools to answer the user's question. "
                "When you have enough information, provide a clear final answer. "
                "Do not make up data — only use what the tools return."
            )},
            {"role": "user", "content": goal},
        ]

        for step in range(1, self.max_steps + 1):
            # ── Think ───────────────────────────────────────
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=TOOL_SCHEMAS,
                tool_choice="auto",
            )

            # Track token usage
            if response.usage:
                self.total_tokens_used += response.usage.total_tokens

            assistant_msg = response.choices[0].message

            # If no tool call → we have a final answer
            if not assistant_msg.tool_calls:
                self.execution_trace.append({
                    "step": step,
                    "type": "final_answer",
                    "content": assistant_msg.content,
                })
                return {
                    "answer": assistant_msg.content,
                    "steps": step,
                    "total_tokens": self.total_tokens_used,
                    "trace": self.execution_trace,
                }

            # ── Act ─────────────────────────────────────────
            # Process all tool calls in this response
            messages.append(assistant_msg)  # add the assistant's tool_call message

            for tool_call in assistant_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Execute tool
                tool_output = self._call_tool(func_name, func_args)

                # Log the step
                self.execution_trace.append({
                    "step": step,
                    "type": "tool_call",
                    "tool": func_name,
                    "input": func_args,
                    "output": tool_output[:300] + ("..." if len(tool_output) > 300 else ""),
                })

                # ── Observe ─────────────────────────────────
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_output,
                })

        # Hit max steps without a final answer
        self.execution_trace.append({
            "step": self.max_steps,
            "type": "budget_exceeded",
            "content": "Reached maximum step limit.",
        })
        return {
            "answer": "I was not able to complete the task within the step budget.",
            "steps": self.max_steps,
            "total_tokens": self.total_tokens_used,
            "trace": self.execution_trace,
        }


print("SimpleAgent class defined.")

## 4. Helper: Display Execution Trace

A utility function to pretty-print the agent's step-by-step reasoning.

In [ ]:
def display_trace(result: dict, task_name: str = ""):
    """Pretty-print an agent execution trace."""
    print("=" * 70)
    if task_name:
        print(f"TASK: {task_name}")
    print(f"Steps: {result['steps']} | Tokens used: {result.get('total_tokens', 'N/A')}")
    print("-" * 70)

    for entry in result["trace"]:
        step = entry["step"]
        if entry["type"] == "tool_call":
            print(f"  Step {step} [TOOL CALL]")
            print(f"    Tool:   {entry['tool']}")
            print(f"    Input:  {json.dumps(entry['input'])}")
            output_preview = entry['output'][:150]
            print(f"    Output: {output_preview}")
        elif entry["type"] == "final_answer":
            print(f"  Step {step} [FINAL ANSWER]")
            print(f"    {entry['content'][:300]}")
        elif entry["type"] == "budget_exceeded":
            print(f"  Step {step} [BUDGET EXCEEDED]")
        print()

    print("=" * 70)

print("display_trace() helper ready.")

## 5. Run the Agent on Three Tasks

We test the agent with tasks of increasing complexity:

1. **Single-tool**: Find products under $50 (uses `search_products`).
2. **Single-tool with math**: Calculate total cost with tax (uses `calculate`).
3. **Multi-step**: Look up a customer, then find products matching their interests
   (uses `lookup_customer` + `search_products`).

Note: If you don't have an API key set, the cells below will show an error message.

In [ ]:
# Task 1: Single-tool product search
agent = SimpleAgent(model="gpt-4o-mini", max_steps=5)
result1 = agent.run("What products do we have under $50?")
display_trace(result1, "Products under $50")

In [ ]:
# Task 2: Calculator task
agent = SimpleAgent(model="gpt-4o-mini", max_steps=5)
result2 = agent.run("Calculate the total cost of 3 widgets at $12.99 each plus 8.5% tax")
display_trace(result2, "Cost calculation with tax")

In [ ]:
# Task 3: Multi-step — lookup customer, then find matching products
agent = SimpleAgent(model="gpt-4o-mini", max_steps=5)
result3 = agent.run(
    "Look up customer C001 and find products they might like based on their interests."
)
display_trace(result3, "Customer lookup + product recommendation")

## 6. Guardrails Summary

Let's review the guardrails we built into the agent and their impact.

In [ ]:
# Summarize guardrails across all three tasks
all_results = [
    ("Products under $50", result1),
    ("Cost calculation", result2),
    ("Customer + products", result3),
]

guardrail_data = []
for name, result in all_results:
    steps = result["steps"]
    tokens = result.get("total_tokens", 0)
    tool_calls = [e for e in result["trace"] if e["type"] == "tool_call"]
    tools_used = set(e["tool"] for e in tool_calls)
    budget_hit = any(e["type"] == "budget_exceeded" for e in result["trace"])

    guardrail_data.append([
        name,
        steps,
        "5",  # max_steps budget
        len(tool_calls),
        ", ".join(sorted(tools_used)) if tools_used else "none",
        tokens,
        "YES" if budget_hit else "no",
    ])

print("=== Guardrails Report ===")
print(tabulate(
    guardrail_data,
    headers=["Task", "Steps Used", "Step Budget", "Tool Calls", "Tools", "Tokens", "Budget Hit?"],
    tablefmt="github",
))

print("\nGuardrails enforced:")
print("  1. Step budget:     max 5 steps per task")
print("  2. Tool allowlist:  only search_products, calculate, lookup_customer")
print("  3. Cost tracking:   total tokens logged per task")

## 7. Testing the Allowlist Guardrail

What happens if we restrict the agent's tools? Let's remove `lookup_customer` from the
allowlist and see how the agent handles a task that requires it.

In [ ]:
# Create an agent with a restricted allowlist
restricted_agent = SimpleAgent(model="gpt-4o-mini", max_steps=5)
restricted_agent.tool_allowlist = {"search_products", "calculate"}  # no lookup_customer

result_restricted = restricted_agent.run(
    "Look up customer C001 and find products they might like."
)
display_trace(result_restricted, "Restricted agent (no customer lookup)")

print("Notice: The agent either adapts its strategy or reports that it cannot")
print("access customer data — the guardrail prevents unauthorized tool use.")

## Key Takeaways

1. **Agents are loops, not magic.** Think → Act → Observe, repeated until done. The LLM
   decides which tool to call and when to stop.

2. **Function calling is the interface.** The tool schemas tell the LLM what's available;
   the LLM returns structured JSON to invoke them.

3. **Guardrails are essential in production:**
   - **Step budgets** prevent runaway loops.
   - **Tool allowlists** enforce the principle of least privilege.
   - **Cost tracking** makes spend visible and auditable.

4. **Start simple.** This notebook's `SimpleAgent` is ~60 lines of code. Frameworks like
   LangChain and CrewAI add convenience, but the core loop is exactly what you see here.

---

*From "AI Enterprise Architect" — Chapter 11: AI Agents and Tool Use*